# T-07: Chunk Size Benchmark

**Goal:** Compare 3 chunking strategies on the Revolut synthetic corpus and pick the best for production.

**Strategies tested:**
1. **3-sentence window** (current production, sentence-based)
2. **2-sentence window** (smaller, more granular)
3. **80-token window** (token-based, sliding window with overlap)

**Metrics:**
- Total chunk count (more chunks = finer granularity, but also more noise)
- Average chunk length in chars and tokens
- Silhouette score on KMeans clustering (k=6, as a stable reference)
- T-05 filter survival rate (how many chunks pass the quality filter)

**Note on baseline:** KMeans is the current production clusterer. T-09 will replace it with BERTopic+HDBSCAN, so silhouette numbers here are a *snapshot* for relative comparison, not absolute quality.

In [1]:
# Setup
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from pathlib import Path
import re
import numpy as np
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from pipeline.chunker import _is_chunk_substantive, _safe_filename

# Make sure punkt is available
try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab', quiet=True)

DATA_DIR = Path('../data/synthetic/revolut')
TXT_FILES = sorted(DATA_DIR.glob('*.txt'))
print(f'Found {len(TXT_FILES)} .txt files in {DATA_DIR}')
for f in TXT_FILES:
    print(f'  {f.name}')

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Found 8 .txt files in ../data/synthetic/revolut
  interview_revolut_01.txt
  interview_revolut_02.txt
  interview_revolut_03.txt
  interview_revolut_04.txt
  sales_notes_revolut.txt
  usability_revolut_01.txt
  usability_revolut_02.txt
  usability_revolut_03.txt


In [2]:
# Strategy 1: N-sentence window (used for strategies 1 and 2)
def chunk_by_sentences(raw_text: str, n_sentences: int) -> list[str]:
    sentences = sent_tokenize(raw_text)
    sentences = [s.strip() for s in sentences if s.strip() and len(s) >= 10]
    chunks = []
    for i in range(0, len(sentences), n_sentences):
        group = sentences[i:i + n_sentences]
        chunks.append(' '.join(group))
    return chunks

# Strategy 3: token-window with sliding overlap
def chunk_by_tokens(raw_text: str, target_tokens: int = 80, overlap: int = 20) -> list[str]:
    tokens = raw_text.split()
    chunks = []
    step = max(target_tokens - overlap, 1)
    for i in range(0, len(tokens), step):
        group = tokens[i:i + target_tokens]
        if len(group) < 10:
            continue
        chunks.append(' '.join(group))
    return chunks

print('Chunking functions ready.')

Chunking functions ready.


In [3]:
# Run all 3 strategies across all .txt files
STRATEGIES = {
    '3-sentence (current)': lambda txt: chunk_by_sentences(txt, 3),
    '2-sentence':           lambda txt: chunk_by_sentences(txt, 2),
    '80-token sliding':     lambda txt: chunk_by_tokens(txt, target_tokens=80, overlap=20),
}

results = {name: [] for name in STRATEGIES}

for fp in TXT_FILES:
    raw = fp.read_text(encoding='utf-8')
    for name, fn in STRATEGIES.items():
        for text in fn(raw):
            results[name].append({'text': text, 'filename': fp.name})

# Basic counts
summary = []
for name, chunks in results.items():
    chars = [len(c['text']) for c in chunks]
    toks  = [len(c['text'].split()) for c in chunks]
    summary.append({
        'strategy': name,
        'total_chunks': len(chunks),
        'avg_chars': round(np.mean(chars), 1),
        'avg_tokens': round(np.mean(toks), 1),
        'median_tokens': int(np.median(toks)),
        'min_tokens': min(toks),
        'max_tokens': max(toks),
    })

pd.DataFrame(summary)

,strategy,total_chunks,avg_chars,avg_tokens,median_tokens,min_tokens,max_tokens
0,3-sentence (current),306,179.4,31.3,28,8,98
1,2-sentence,459,119.2,20.9,17,3,87
2,80-token sliding,164,444.0,77.7,80,12,80


In [4]:
# Apply T-05 filter to each strategy — how many chunks survive?
filter_summary = []
for name, chunks in results.items():
    before = len(chunks)
    after = sum(1 for c in chunks if _is_chunk_substantive(c['text']))
    filter_summary.append({
        'strategy': name,
        'before_t05': before,
        'after_t05': after,
        'filtered_pct': round(100 * (before - after) / before, 1) if before else 0,
    })

pd.DataFrame(filter_summary)

,strategy,before_t05,after_t05,filtered_pct
0,3-sentence (current),306,272,11.1
1,2-sentence,459,286,37.7
2,80-token sliding,164,160,2.4


In [5]:
# Embed and cluster each strategy with KMeans (k=6, fixed for comparability)
model = SentenceTransformer('all-MiniLM-L6-v2')
K = 6

cluster_summary = []
for name, chunks in results.items():
    # Apply T-05 filter first — clustering should run on production-like input
    filtered = [c for c in chunks if _is_chunk_substantive(c['text'])]
    texts = [c['text'] for c in filtered]
    if len(texts) < K + 1:
        cluster_summary.append({'strategy': name, 'silhouette': None, 'n_chunks': len(texts), 'note': 'too few chunks for clustering'})
        continue
    
    embeddings = model.encode(texts, show_progress_bar=False)
    km = KMeans(n_clusters=K, random_state=42, n_init='auto').fit(embeddings)
    sil = silhouette_score(embeddings, km.labels_, metric='cosine')
    cluster_summary.append({
        'strategy': name,
        'n_chunks_clustered': len(texts),
        'silhouette_cosine': round(sil, 4),
        'k': K,
    })

pd.DataFrame(cluster_summary)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10814.66it/s]


,strategy,n_chunks_clustered,silhouette_cosine,k
0,3-sentence (current),272,0.0838,6
1,2-sentence,286,0.0761,6
2,80-token sliding,160,0.1103,6


## Findings

**Winning strategy:** 80-token sliding window (overlap = 20)

### Quantitative evidence

| Strategy | Chunks (raw) | Avg tokens | Filter drop | Chunks (clustered) | Silhouette |
|---|---|---|---|---|---|
| 3-sentence (current) | 306 | 31.3 | 11.1% | 272 | 0.0838 |
| 2-sentence | 459 | 20.9 | 37.7% | 286 | 0.0761 |
| **80-token sliding** | **164** | **77.7** | **2.4%** | **160** | **0.1103** |

### Reasoning

- **Silhouette improvement: +31.6%** (0.1103 vs 0.0838). Wider than expected — KMeans clusters genuinely benefit from uniform chunk size.
- **Lower memory footprint:** 160 chunks vs 272 for 3-sentence. With Streamlit's 500-chunk ceiling, this gives generous headroom.
- **Higher T-05 survival:** 2.4% dropped vs 11.1%. Token-based chunks rarely fall below the 15-token / 2-sentence floor.
- **Stable chunk size:** min=12, median=80, max=80. Sentence-based had min=8, max=98 — variance hurts embedding consistency.

**Why 2-sentence is the loser:** Filter drops 37.7% — half the chunks waste downstream compute. Silhouette also lowest (0.0761).

### Recommendation

Migrate `chunker.py` to 80-token sliding window (overlap = 20). Sentence-based logic kept as a clear fallback path in case the production data distribution shifts.

### Caveat

Numbers are measured on KMeans + cosine, which T-09 (BERTopic + HDBSCAN) will replace. The relative ranking should hold (density-based clustering also prefers uniform chunk sizes), but re-run this benchmark after T-09 lands to confirm.

**Decision logged in:** `docs/decisions.md` (pending Lucas sign-off on this PR).